# GCP Pose Estimation — Kaggle Training

## Before running: download the data as a ZIP and upload to Kaggle

**1. Download the Drive folder as ZIP (on your laptop)**
- Open [drive.google.com](https://drive.google.com) → find the shared GCP data folder
- Right-click the folder → **Download**
- Chrome will zip it and download. If the folder is >2 GB, Drive splits it into multiple ZIPs — download **all** of them.

**2. Upload the ZIP(s) as a Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload all ZIP file(s) → title: `skylark-gcp-data` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `skylark-gcp-data` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [ ]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Wire config to Kaggle input dataset
import os, json, yaml, glob

# Find gcp_marks.json anywhere under /kaggle/input
hits = glob.glob('/kaggle/input/**/gcp_marks.json', recursive=True)
assert hits, 'gcp_marks.json not found under /kaggle/input — add skylark-gcp-data as notebook input'
label_file_src = hits[0]
BASE = os.path.dirname(label_file_src)
print('Found dataset at:', BASE)
print('Contents:', sorted(os.listdir(BASE)))

def find_dir(base, *candidates):
    for name in candidates:
        p = os.path.join(base, name)
        if os.path.isdir(p): return p
    raise FileNotFoundError(f'None of {candidates} found in {base}')

train_dir = find_dir(BASE, 'train_clean', 'train_dataset', 'train')
test_dir  = find_dir(BASE, 'test_clean',  'test_dataset',  'test')
print(f'train_dir : {train_dir}')
print(f'test_dir  : {test_dir}')

# Copy + patch label keys ('&' -> 'and') into writable data/
os.makedirs('data', exist_ok=True)
raw = json.load(open(label_file_src))
patched = {k.replace('&', 'and'): v for k, v in raw.items()}
local_labels = 'data/gcp_marks.json'
json.dump(patched, open(local_labels, 'w'))
n_fixed = sum(1 for k in raw if '&' in k)
if n_fixed: print(f'Patched {n_fixed} label keys (& -> and)')

# Wire config
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': local_labels,
                'train_dir':  train_dir,
                'test_dir':   test_dir,
                'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))
print('Config updated.')

# Verify coverage
labels = json.load(open(local_labels))
found  = sum(1 for k in labels if os.path.exists(os.path.join(train_dir, k)))
pct    = 100 * found // len(labels)
print(f'Train images on disk: {found}/{len(labels)} ({pct}%)')
print('OK — enough to train.' if found >= len(labels) * 0.8 else
      'WARNING: <80% images — check folder names above.')


In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: restore 40 epochs
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 40
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

## Artifacts to download

From the Kaggle Output panel (right sidebar), download these three files:

- `gcp/outputs/best.pt` — model weights (upload to Drive, copy the shareable link)
- `gcp/outputs/training_log.csv` — open it, read the last row for PCK@10/25/50 + Macro-F1
- `gcp/predictions.json` — this is your submission file for Skylark

Then update `README.md` in the repo:
1. Fill in the Results table with the PCK/F1 numbers
2. Add the Drive link for `best.pt`
3. `git add README.md && git commit -m "docs: add training results and weights link" && git push`